In [1]:
import os
os.chdir(r"C:\Users\Noel\Documents\Personal Projects\f1-rag-knowledge-assistant")
print(os.getcwd())

C:\Users\Noel\Documents\Personal Projects\f1-rag-knowledge-assistant


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA


In [3]:
# Define the path to your F1 PDF
PDF_PATH = "C:/Users/Noel/Documents/Personal Projects/f1-rag-knowledge-assistant/data/F1_2024_BritishGP_RaceReport.pdf"

# Create the loader
loader = PyPDFLoader(PDF_PATH)

# Load the document pages
pages = loader.load()

# See what we got
print(f"Total pages loaded: {len(pages)}")
print(f"\nFirst page preview:\n{pages[0].page_content[:500]}")

Total pages loaded: 7

First page preview:
FIA FORMULA ONE WORLD CHAMPIONSHIP
 
 2024 FORMULA 1
QATAR AIRWAYS
BRITISH GRAND PRIX
 OFFICIAL RACE REPORT & STATISTICAL REVIEW
 
Circuit
Silverstone Circuit, Northamptonshire, United Kingdom
Race Date
Sunday, 7 July 2024
Round
Round 12 of 24 — 2024 FIA Formula One World Championship
Distance
52 laps × 5.891 km = 306.198 km
Weather
Partly cloudy, 21°C ambient, 38°C track surface
Attendance
480,000 (race weekend total — record for Silverstone)
Document Ref
F1-2024-GBR-RACE-REPORT-v1.2
This docum


In [4]:
#Create the splitters
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

#Split the pages into chunks
chunks = splitter.split_documents(pages)

print(f"total chunks created :{len(chunks)}")
print(f"\nFirst chunk preview:\n{chunks[0].page_content}")

total chunks created :30

First chunk preview:
FIA FORMULA ONE WORLD CHAMPIONSHIP
 
 2024 FORMULA 1
QATAR AIRWAYS
BRITISH GRAND PRIX
 OFFICIAL RACE REPORT & STATISTICAL REVIEW
 
Circuit
Silverstone Circuit, Northamptonshire, United Kingdom
Race Date
Sunday, 7 July 2024
Round
Round 12 of 24 — 2024 FIA Formula One World Championship
Distance
52 laps × 5.891 km = 306.198 km
Weather
Partly cloudy, 21°C ambient, 38°C track surface
Attendance
480,000 (race weekend total — record for Silverstone)
Document Ref
F1-2024-GBR-RACE-REPORT-v1.2


In [5]:
import shutil
if os.path.exists("chroma_db"):
    shutil.rmtree("chroma_db")
    print("Old ChromaDB cleared!")

    
#Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

#create ChromaDB and store chunks
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "chroma_db"
)

print(f"Total vectors stored :{vector_store._collection.count()}")


Old ChromaDB cleared!


C:\Users\Noel\AppData\Local\Temp\ipykernel_29620\79521472.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\Noel\Documents\Personal Projects\f1-rag-knowledge-assistant\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but

Total vectors stored :30


In [14]:
#Load the LLM
llm = Ollama(model="llama3.2")

#Build the RAG Chain
qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = vector_store.as_retriever(
        search_type= "mmr",
        search_kwargs={
            "k": 5,
            "fetch_k": 15
        }
    ),
    chain_type = "stuff"
)

print("RAG chain ready!")

RAG chain ready!


In [15]:
#Ask the question
question = "Who won the 2024 British Grand Prix and by how much?"
result = qa_chain.invoke({"query":question})
print(result["result"])

According to the official race report, Lando Norris won the 2024 British Grand Prix with a margin of 7.034 seconds from the second-place finisher.


In [17]:
# Try a more specific question
question2 = "Which driver crossed the finish line first at Silverstone in 2024?"
result2 = qa_chain.invoke({"query": question2})
print(result2["result"])

The answer is not explicitly stated in the provided context, but based on the information given, Lando Norris secured pole position and went on to win the race with a significant margin of 7.034 seconds. It can be inferred that he crossed the finish line first at Silverstone in 2024.


In [18]:
question = "Who won the 2024 British Grand Prix?"
result = qa_chain.invoke({"query": question})
print(result["result"])

According to the context, Lando Norris (McLaren) secured pole position and took the chequered flag, winning the 2024 British Grand Prix by a margin of 7.034 seconds.


"I improved retrieval quality by switching from cosine similarity search to Maximum Marginal Relevance, which fetches a larger candidate pool and selects diverse chunks — this solved a problem where semantically similar but contextually wrong chunks were being returned."

In [20]:
question = [
    "What was the fastest lap of the race?",
    "Why did Tsunoda get a penalty?",
    "What tyres did Norris use during the race?"
]

for q in question:
    result = qa_chain.invoke({"query":q})
    print(f"Q: {q}")
    print(f"A: {result['result']}\n")

Q: What was the fastest lap of the race?
A: The fastest lap of the race was recorded at Lap 47 with a time of 1:27.097, achieved by Lando Norris. This is also marked as the NOR FASTEST LAP of race.

Q: Why did Tsunoda get a penalty?
A: Tsunoda failed to respect the pit entry yellow flag marshalling zone at the exit of Club Corner, passing Car #23 (Albon) under yellow conditions in contravention of Sporting Regulation 38.4.

Q: What tyres did Norris use during the race?
A: According to the provided context, Norris used Medium (C3) tyres for most of the race and Hard (C2) tyres later on.

